In [14]:
#load all the necessary libraries
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from rdkit.Chem import Descriptors


In [ ]:
#Datasets
#load the datasets 
df_delaney = pd.read_csv(Path("data/delaney.csv"))
df_mic = pd.read_csv(Path("data/ecoli_mics_sample.csv"))

#inspect the datasets
print("head delaney:",df_delaney.head())
print("head mic:", df_mic.head())
print("delaney statistics:",df_delaney.describe())
print("mic statistics:",df_mic.describe())

#check for missing values and duplicates: 
print("duplicates and missing values in delaney:",df_delaney.duplicated().sum(), df_delaney.isnull().sum()) #no duplicates or missing values in delaney dataset
print("duplicates and missing values in mic:", df_mic.duplicated().sum(), df_mic.isnull().sum()) #two missing values in mic dataset, no duplicates

#check if units for mic are consistent:
print("units in mic dataset:", df_mic['unit'].unique()) #units are consistent. 

#drop the missing values in mic dataset
df_mic.dropna(inplace=True)
df_mic.info()

head delaney:                       Compound ID  logS               smiles
0       1,1,1,2-Tetrachloroethane -2.18       ClCC(Cl)(Cl)Cl
1           1,1,1-Trichloroethane -2.00         CC(Cl)(Cl)Cl
2       1,1,2,2-Tetrachloroethane -1.74       ClC(Cl)C(Cl)Cl
3           1,1,2-Trichloroethane -1.48           ClCC(Cl)Cl
4  1,1,2-Trichlorotrifluoroethane -3.04  FC(F)(Cl)C(F)(Cl)Cl
head mic:        chembl_id                                             smiles    mic  \
0  CHEMBL1689756  CN(C)[C@@H]1C(O)=C(C(N)=O)C(=O)[C@@]2(O)C(O)=C...   0.25   
1  CHEMBL1222951  CO/N=C1/CCN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3...   8.00   
2  CHEMBL2403434          COc1cccc(CC(=O)Nc2nc(-c3ccc(Br)cc3)cs2)c1  12.50   
3      CHEMBL148  C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(SCCNC=N)C[C...   0.19   
4       CHEMBL22                    COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC   0.25   

      unit  
0  ug.mL-1  
1  ug.mL-1  
2  ug.mL-1  
3  ug.mL-1  
4  ug.mL-1  
delaney statistics:               logS
count  1144.000000
mean 

In [22]:
# function for adding images from RDKit to the datasets:
def add_molecule_images(df): 
    mol_list = df['smiles'].apply(lambda s: Chem.MolFromSmiles(s))
    df['image']= mol_list.apply(lambda m: Chem.Draw.MolToImage(m, size=(200, 200)))
    return df

In [26]:
# function for adding fingerprint column to the datasets:
from rdkit.Chem import rdFingerprintGenerator, DataStructs
def add_fingerprits(df): 
    mol_list = Chem.MolFromSmiles(df['smiles'])
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1800)
    df['fingerprint'] = mfpgen.GetFingerprint(mol_list)
    return df

In [27]:
#test: 
df_delaney_images = add_molecule_images(df_delaney)
#print(df_delaney_images.head())

df_delaney_fingerprints = add_fingerprits(df_delaney)
print(df_delaney_fingerprints.head())

TypeError: No registered converter was able to produce a C++ rvalue of type class std::basic_string<wchar_t,struct std::char_traits<wchar_t>,class std::allocator<wchar_t> > from this Python object of type Series